# اسباق 12 - ایجنٹ اسکریچ پیڈ کے ساتھ چیٹ ہسٹری کی کمی

یہ نوٹ بک مائیکروسافٹ ایجنٹ فریم ورک کے ذریعے طویل بات چیت میں سیاق و سباق کو کیسے سنبھالنا ہے دکھاتی ہے۔ جیسے جیسے بات چیت بڑھتی ہے، ٹوکن کی تعداد بڑھتی جاتی ہے — آخرکار ماڈل کی سیاق و سباق کی کھڑکی سے تجاوز کر جاتی ہے۔ ہم اس مسئلے کو **سیاق و سباق کے خلاصے کے نمونے** اور **ایجنٹ اسکریچ پیڈ** کے ذریعے مستقل یادداشت کے ساتھ حل کرتے ہیں۔

## آپ کیا سیکھیں گے:
1. **سیاق و سباق کا انتظام کیوں اہم ہے**: ٹوکن کی حدوں اور سیاق و سباق کی کھڑکیوں کو سمجھنا
2. **سیاق و سباق سے آگاہ ایجنٹس**: ایسے ایجنٹس بنانا جو اپنی بات چیت کے سیاق و سباق کو خود سنبھالیں
3. **سیاق و سباق کے خلاصے کا نمونہ**: بات چیت کی تاریخ کو مختصر کرنے کے لئے اوزار استعمال کرنا
4. **ایجنٹ اسکریچ پیڈ**: مستقل یادداشت جو سیاق و سباق کی کمی کے باوجود برقرار رہے

## ضروریات:
- ماحول کے متغیرات کے ساتھ Azure OpenAI سیٹ اپ کیا ہوا
- پہلے کے اسباق سے بنیادی ایجنٹ تصورات کی سمجھ


## ترتیب


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## تناظر کے انتظام کی اہمیت کیوں ہے

ہر LLM کی ایک محدود **تناظر ونڈو** ہوتی ہے — زیادہ سے زیادہ ٹوکنز کی تعداد جو وہ ایک درخواست میں پروسیس کر سکتا ہے۔ جیسے جیسے ایک کثیر مرحلہ گفتگو آگے بڑھتی ہے:

- **ٹوکن کی تعداد خطی طور پر بڑھتی ہے** ہر صارف کے پیغام اور معاون کے جواب کے ساتھ۔
- **پرومپٹ ٹوکنز قیمت میں غالب ہوتے ہیں** کیونکہ پوری تاریخ ہر مرحلہ پر دوبارہ بھیجی جاتی ہے۔
- آخرکار گفتگو **تناظر ونڈو سے تجاوز کر جاتی ہے** اور ماڈل یا تو کاٹ دیتا ہے یا غلطی دیتا ہے۔

### تناظر کے انتظام کی حکمت عملیاں

| حکمت عملی | یہ کیسے کام کرتی ہے | فائدہ اور نقصان |
|---|---|---|
| **کٹاؤ** | قدیم پیغامات کو چھوڑ دینا | ابتدائی تناظر ضائع ہو جاتا ہے |
| **خلاصہ سازی** | پرانے پیغامات کو مختصر کر کے خلاصہ بنانا | کچھ تفصیل ضائع ہوتی ہے، لیکن اہم نکات بنے رہتے ہیں |
| **اسکریچ پیڈ / خارجی میموری** | گفتگو کے باہر اہم حقائق ذخیرہ کرنا | ٹول کالز کی ضرورت ہوتی ہے، لیکن کسی بھی کمی برداشت کر لیتا ہے |

اس نوٹ بک میں ہم **خلاصہ سازی** کو **اسکریچ پیڈ ٹول** کے ساتھ ملا کر استعمال کرتے ہیں تاکہ ایجنٹ گفتگو کی تسلسل برقرار رکھ سکے حتیٰ کہ جب گفتگو کا تاریخی خلاصہ کیا جائے۔


## تناظر سے آگاہ ایجنٹ کی تخلیق


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## طویل گفتگو کا تمثیل

آئیے ایک کثیر دورانیہ گفتگو کا جائزہ لیتے ہیں تاکہ دیکھا جا سکے کہ سیاق و سباق کیسے جمع ہوتا ہے۔ نمائندہ کو اہم تفصیلات (پسند، بجٹ، سفر کی تاریخیں) کو دوروں کے درمیان محفوظ رکھنا چاہیے اور تسلسل کا مظاہرہ کرنا چاہیے۔


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

نوٹ کریں کہ ایجنٹ پہلے کے دوروں سے سیاق و سباق کو برقرار رکھتا ہے — اسے جاپان، سوشی، مندر، فوٹوگرافی، 3000 ڈالر کے بجٹ، اکیلے سفر، اور اپریل کے وقت کے بارے میں معلوم ہے۔ ایک مختصر گفتگو میں یہ اچھی طرح کام کرتا ہے، لیکن جیسے جیسے گفتگو بڑھتی ہے مکمل تاریخ کو دوبارہ بھیجنا مہنگا ہو جاتا ہے۔

آئیے مزید دوروں کے ساتھ گفتگو جاری رکھیں تاکہ سیاق و سباق کے جمع ہونے کو دیکھ سکیں:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## سیاق و سباق کا خلاصہ کرنے کا نمونہ

جوں جوں بات چیت بڑھتی ہے، ہم **خلاصہ کرنے کے آلے** کو استعمال کر سکتے ہیں تاکہ جمع شدہ سیاق و سباق کو ایک مختصر شکل میں تبدیل کیا جا سکے۔ ایجنٹ اس آلے کو بنیادی ترجیحات کو ریکارڈ کرنے کے لیے کال کرتا ہے تاکہ اگر پرانے پیغامات ہٹا بھی دیے جائیں، تو اہم معلومات محفوظ رہیں۔

یہ نمونہ زیادہ پیچیدہ تاریخ کی کمی کے لیے بنیادی بلاک ہے:
1. ایجنٹ بات چیت سے اہم حقائق کی شناخت کرتا ہے
2. یہ خلاصہ کرنے کے آلے کو انہیں محفوظ کرنے کے لیے کال کرتا ہے
3. پرانے پیغامات کو محفوظ طریقے سے ہٹایا جا سکتا ہے کیونکہ خلاصہ وہ چیزیں پر محیط ہوتا ہے جو اہم ہیں

نیچے ہم ایک `summarize_preferences` آلے کی تعریف کرتے ہیں جسے ایجنٹ اس بات کا مختصر خلاصہ ریکارڈ کرنے کے لیے کال کر سکتا ہے جو اس نے سیکھا ہے۔


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## خلاصہ

اس سبق میں آپ نے سیکھا کہ Microsoft Agent Framework کا استعمال کرتے ہوئے طویل مدتی ایجنٹ گفتگو میں سیاق و سباق کو کیسے منظم کیا جائے:

### اہم تصورات
- **سیاق و سباق کی ونڈوز محدود ہوتی ہیں** — گفتگو کی تاریخ میں ہر ٹوکن کی قیمت ہوتی ہے اور یہ حد میں شمار ہوتا ہے۔
- **خلاصہ سازی کے اوزار** ایجنٹ کو جمع شدہ سیاق و سباق کو مختصر خلاصوں میں تبدیل کرنے دیتے ہیں، جس سے ٹوکن کے استعمال میں کمی آتی ہے جبکہ ضروری معلومات برقرار رہتی ہیں۔
- **ایجنٹ اسکریچ پیڈز** مستقل خارجی میموری فراہم کرتے ہیں جو کسی بھی گفتگو کی کمی کو برداشت کر لیتے ہیں۔

### آپ نے کیا بنایا
- ایک **سیاق و سباق کے لحاظ سے باخبر ایجنٹ** جو کثیر موڑ والی گفتگو میں تسلسل قائم رکھتا ہے
- ایک **خلاصہ سازی کا آلہ** (`summarize_preferences`) جو صارف کی اہم تفصیلات کو مختصر شکل میں ریکارڈ کرتا ہے
- ایک **کثیر موڑ والی گفتگو** جو سیاق و سباق کی بحالی اور تبدیلی کے انتظام کو ظاہر کرتی ہے

### حقیقی دنیا میں استعمال
- **کسٹمر سروس بوٹس**: طویل سپورٹ سیشنز میں ترجیحات کو یاد رکھیں
- **ذاتی معاونین**: جاری منصوبوں کو بغیر سیاق و سباق دوبارہ سمجھائے ٹریک کریں
- **تعلیمی ٹیوٹرز**: بہت سی بات چیت کے دوران طالب علم کی پیشرفت برقرار رکھیں

### اگلے مرحلے
- فائل پر مبنی مستقل مزاجی کے ساتھ مکمل اسکریچ پیڈ ٹول تیار کریں
- خلاصہ سازی کے بعد خودکار تاریخ کو کم کرنے کا اضافہ کریں
- معنوی میموری تلاش کے لیے ویکٹر ڈیٹا بیسز کے ساتھ ملائیں
- ایجنٹس بنائیں جو مکمل سیاق و سباق کے ساتھ دنوں بعد بات چیت دوبارہ شروع کر سکیں


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
